# 02. User Features

## Objective
Aggregate event-level data into one row per user with meaningful behavioral metrics.

## Steps Performed
1. Loaded cleaned dataset (74,817 rows)
2. Aggregated per user: total events, sessions, purchases, spend, unique events, first and last activity
3. Calculated active days (last activity - first activity)
4. Calculated purchase frequency (total purchases / active days)
5. Validated output - zero nulls, zero negative values
6. Saved to data/cleaned/

## Output
- User features dataset: 1,000 rows × 10 columns
- Average active days: ~200
- Average purchases per user: 10.68
- Purchase frequency range: 0.015 to 0.108

In [1]:
import pandas as pd
import numpy as np


#cleaned dataset
df = pd.read_csv(r'F:\Projects\User Retention Study\User Churn Analysis\Data\cleaned\ecommerce_cleaned.csv')

#Converting the time stamp
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

#check
print(df.shape)
print(df.dtypes)

(74817, 6)
UserID                int64
SessionID             int64
Timestamp    datetime64[us]
EventType               str
ProductID               str
Amount              float64
dtype: object


In [5]:
user_features = df.groupby('UserID').agg(
    total_events = ('EventType', 'count'),
    total_sessions = ('SessionID', 'nunique'), 
    total_purchases = ('EventType', lambda x: (x == 'purchase').sum()),
    total_spend = ('Amount', 'sum'),
    unique_events = ('EventType', 'nunique'),
    first_activity = ('Timestamp', 'min'),
    last_activity = ('Timestamp', 'max')
    
).reset_index()


#check
print(user_features.shape)
print(user_features.head())

(1000, 8)
   UserID  total_events  total_sessions  total_purchases  total_spend  \
0       1            82              10                8  1996.128780   
1       2            73              10               13  2676.553720   
2       3            64              10                6  1512.699392   
3       4            83              10                9  2322.826358   
4       5            84              10                7  2189.316345   

   unique_events             first_activity              last_activity  
0              7 2024-01-01 23:09:51.956825 2024-07-22 20:10:14.181302  
1              7 2024-01-01 14:07:40.491141 2024-07-22 08:54:54.594265  
2              7 2024-01-05 15:44:25.330449 2024-07-23 17:38:50.228803  
3              7 2024-01-01 02:11:06.298369 2024-07-23 19:57:57.196352  
4              7 2024-01-01 14:03:07.709158 2024-07-24 10:05:47.264077  


In [8]:
#How long they been active
user_features['active_days'] = (
    user_features['last_activity'] - user_features['first_activity']
).dt.days

#frequency of purchase

user_features['purchase_frequency'] =(
    user_features['total_purchases'] / user_features['active_days']
).round(4)

#check
print(user_features[['UserID', 'active_days', 'purchase_frequency']].head(10))

   UserID  active_days  purchase_frequency
0       1          202              0.0396
1       2          202              0.0644
2       3          200              0.0300
3       4          204              0.0441
4       5          204              0.0343
5       6          200              0.0800
6       7          192              0.0625
7       8          199              0.0603
8       9          199              0.0804
9      10          204              0.0882


In [10]:
#checking

print("Shape:", user_features.shape)

print("\nNull values:")
print(user_features.isnull().sum())

## Check for impossible values
print("\nNegative active days:", (user_features['active_days']< 0).sum())
print("Negative spend: ", (user_features['total_spend']<0).sum())

# Summary statistics
print("\nKey status:")
print(user_features[['total_events', 'total_purchases', 'total_spend', 'active_days', 'purchase_frequency']].describe())

Shape: (1000, 10)

Null values:
UserID                0
total_events          0
total_sessions        0
total_purchases       0
total_spend           0
unique_events         0
first_activity        0
last_activity         0
active_days           0
purchase_frequency    0
dtype: int64

Negative active days: 0
Negative spend:  0

Key status:
       total_events  total_purchases  total_spend  active_days  \
count    1000.00000      1000.000000  1000.000000   1000.00000   
mean       74.81700        10.682000  2704.572999    199.77200   
std         5.47389         3.158933   914.071779      3.89425   
min        58.00000         3.000000   347.149414    178.00000   
25%        71.00000         8.000000  2007.448495    198.00000   
50%        75.00000        10.000000  2649.110063    201.00000   
75%        79.00000        13.000000  3300.153153    202.25000   
max        94.00000        22.000000  6732.148219    205.00000   

       purchase_frequency  
count         1000.000000  
mean   

In [11]:
#save the folder
user_features.to_csv(r'F:\Projects\User Retention Study\User Churn Analysis\Data\cleaned\user_features.csv', index=False)

print(f"Final shape: {user_features.shape}")

Final shape: (1000, 10)
